In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
len(words)

32033

In [5]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [9]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
  
  print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [10]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [11]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]])

In [12]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [23]:
# Create a 2d dimensional embedding space
C = torch.randn((27, 2))
emb = C[X]
emb

tensor([[[ 0.1419,  1.9394],
         [ 0.1419,  1.9394],
         [ 0.1419,  1.9394]],

        [[ 0.1419,  1.9394],
         [ 0.1419,  1.9394],
         [ 1.8484, -0.3307]],

        [[ 0.1419,  1.9394],
         [ 1.8484, -0.3307],
         [-0.1273,  0.4415]],

        [[ 1.8484, -0.3307],
         [-0.1273,  0.4415],
         [-0.1273,  0.4415]],

        [[-0.1273,  0.4415],
         [-0.1273,  0.4415],
         [ 1.4631,  0.6478]],

        [[ 0.1419,  1.9394],
         [ 0.1419,  1.9394],
         [ 0.1419,  1.9394]],

        [[ 0.1419,  1.9394],
         [ 0.1419,  1.9394],
         [ 0.8155, -0.4247]],

        [[ 0.1419,  1.9394],
         [ 0.8155, -0.4247],
         [-0.0840, -0.4230]],

        [[ 0.8155, -0.4247],
         [-0.0840, -0.4230],
         [ 0.2467, -0.4568]],

        [[-0.0840, -0.4230],
         [ 0.2467, -0.4568],
         [ 0.7402,  1.0436]],

        [[ 0.2467, -0.4568],
         [ 0.7402,  1.0436],
         [ 0.2467, -0.4568]],

        [[ 0.7402,  1

In [42]:
# hidden layer: 6 inputs, 100 neurons
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [43]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # use emb.view to manipulate shape of embedding array into one suitable for our nn, emb.shape[0] == -1
h.shape

torch.Size([32, 100])

In [44]:
# concatenating the 6 inputs that represent the 2d embedding of each character in our 3 character context window
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1).shape # can't generalize
torch.cat(torch.unbind(emb, 1), 1).shape # inside exactly same as above

torch.Size([32, 6])

In [45]:
# Output layer: 100 inputs from prev layer, and 27 outputs representing 27 possible next characters
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [46]:
logits = h @ W2 + b2

In [47]:
logits.shape

torch.Size([32, 27])

In [50]:
counts = logits.exp()

In [51]:
prob = counts / counts.sum(1, keepdims=True)

In [52]:
prob.shape

torch.Size([32, 27])

In [54]:
# calculating loss
loss = - prob[torch.arange(32), Y].log().mean()
loss

tensor(15.5450)

In [56]:
# SUMMARY
X.shape, Y.shape # dataset
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [57]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [58]:
emb = C[X] # 32, 3, 2
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
counts = logits.exp()
prob = counts / counts.sum(1, keepdims=True)
loss = - prob[torch.arange(32), Y].log().mean()
loss

tensor(17.7697)